# Amaliy mashgʻulot 1. Koʻp maʼnolilik, kontekst va tillar tipologiyasi

Laboratoriyada kontekst soʻz maʼnolari ehtimolini qanday oʻzgartirishini koʻrdingiz. Bu yerda xuddi shuni kodda bajaramiz:

1. entropiyani hisoblaymiz va u kontekst bilan qanday oʻzgarishini koʻramiz;
2. sodda Bayes klassifikatorini kichik korpus boʻyicha «koʻk» soʻzining maʼnolarini farqlashga oʻrgatamiz;
3. oʻzbek soʻzlarini qoidalar asosida morfemalarga ajratamiz va morfologik koʻp maʼnolilikni topamiz;
4. rus, oʻzbek va ingliz matnlarini solishtiramiz: soʻzlar uzunligi va shakllar soni.

*«NLP algoritmlari» kursining 1-laboratoriyasiga amaliy mashgʻulot. Kataklarni yuqoridan pastga ishga tushiring: Shift+Enter yoki «Runtime → Run all». Javoblaringizni saqlash uchun «File → Save a copy in Drive» ni bosing.*

## 1. Entropiya: noaniqlik oʻlchovi

«Koʻk» soʻzining bir nechta maʼnosi bor. Har bir maʼnoga boshlangʻich «ball» (logit) beramiz, kontekstdagi har bir soʻz ballga oʻz vaznini qoʻshadi, softmax esa ballarni ehtimollarga aylantiradi. Bu laboratoriyadagi model bilan bir xil.

In [ ]:
import numpy as np

def softmax(z):
    z = np.array(z, dtype=float)
    e = np.exp(z - z.max())
    return e / e.sum()

def entropy(p):
    p = np.array(p, dtype=float)
    p = p[p > 0]
    return float(-(p * np.log2(p)).sum())

senses = ["rang: moviy", "koʻkat", "osmon"]
prior = [0.7, 0.3, -0.6]
cue_weights = {
    "koʻylak":           [2.2, -1.0, -0.8],
    "bozordan bir bogʻ": [-0.8, 2.4, -1.0],
    "somsa":             [-0.6, 2.4, -1.0],
    "bahor":             [0.2,  1.0,  0.3],
    "yulduzlar":         [0.1, -1.2,  2.4],
}

def show(context):
    z = np.array(prior, dtype=float)
    for c in context:
        z += cue_weights[c]
    p = softmax(z)
    print("kontekst:", context or "—")
    for s, x in sorted(zip(senses, p), key=lambda t: -t[1]):
        print(f"   {s:<14} {x:6.1%}")
    print(f"   entropiya = {entropy(p):.2f} bit (maksimum {np.log2(len(p)):.2f})\n")

show([])
show(["bahor"])
show(["bahor", "bozordan bir bogʻ"])
show(["bahor", "bozordan bir bogʻ", "somsa"])

## 2. Soʻz maʼnosini kontekstdan aniqlash: sodda Bayes

Haqiqiy vaznlarni hech kim qoʻlda bermaydi: ular har bir qoʻllanishda maʼnosi belgilangan korpusdan oʻrganiladi. Quyida kichik belgilangan korpus: har bir maʼnoga 6 tadan gap.

Morfologiyaga eʼtibor bering: «koʻk», «koʻkka», «koʻkda», «koʻkni» turli qatorlar. Ikki xil belgini solishtiramiz: soʻzlarning oʻzi va dastlabki 5 harfgacha qisqartirilgan soʻzlar (lemmatizatsiyaning qoʻpol oʻrinbosari).

In [ ]:
train = [
    ("U koʻk koʻylak kiyib keldi", "rang"),
    ("Qizning koʻzlari koʻk edi", "rang"),
    ("Devorni koʻk rangga boʻyashdi", "rang"),
    ("Koʻk ruchka bilan yozing", "rang"),
    ("Dengiz suvi koʻk boʻlib koʻrinardi", "rang"),
    ("Bayroqda koʻk rang ham bor", "rang"),
    ("Bozordan bir bogʻ koʻk olib keldim", "koʻkat"),
    ("Bahorda koʻk somsa pishiramiz", "koʻkat"),
    ("Oshga koʻk sepib berishdi", "koʻkat"),
    ("Koʻkni yuvib mayda toʻgʻradi", "koʻkat"),
    ("Bahorgi koʻk juda arzon ekan", "koʻkat"),
    ("Salatga koʻk qoʻshsangiz mazali boʻladi", "koʻkat"),
    ("Koʻkda yulduzlar charaqlardi", "osmon"),
    ("Qushlar koʻkka parvoz qildi", "osmon"),
    ("Koʻk yuzida bulut yoʻq edi", "osmon"),
    ("Tong otib koʻk yorishdi", "osmon"),
    ("Shoir koʻkka boqib sheʼr yozdi", "osmon"),
    ("Samolyot koʻkda gʻoyib boʻldi", "osmon"),
]
test = [
    ("Somsa uchun bozordan koʻk oldim", "koʻkat"),
    ("Koʻkda oy chiqdi", "osmon"),
    ("Koʻk koʻylak sotib oldi", "rang"),
    ("Koʻk juda chiroyli edi", "?"),
]

In [ ]:
import re
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB

def tokens(text):
    return re.findall(r"[а-яёa-zʻʼ'-]+", text.lower())

def stem5(text):
    # lemmatizatsiyaning qoʻpol oʻrinbosari: har bir soʻzning dastlabki 5 harfi
    return " ".join(w[:5] for w in tokens(text))

X_text = [s for s, _ in train]
y = [c for _, c in train]

for name, prep in [("soʻzlarning oʻzi", lambda s: " ".join(tokens(s))), ("dastlabki 5 harf", stem5)]:
    vec = CountVectorizer(preprocessor=prep, token_pattern=r"[^ ]+")
    X = vec.fit_transform(X_text)
    nb = MultinomialNB(alpha=1.0).fit(X, y)
    print(f"=== {name}: lugʻat {len(vec.vocabulary_)} ta belgi")
    for s, gold in test:
        p = nb.predict_proba(vec.transform([s]))[0]
        best = nb.classes_[p.argmax()]
        dist = ", ".join(f"{c} {x:.0%}" for c, x in zip(nb.classes_, p))
        print(f"  {s:<34} → {best:<7} [{dist}]  H={entropy(p):.2f}  (toʻgʻrisi: {gold})")
    print()

Qaysi soʻzlar har bir maʼno uchun eng kuchli «ovoz» beradi? Har bir sinfdagi soʻzlar ehtimolining logarifmlarini koʻramiz («dastlabki 5 harf» belgili model).

In [ ]:
vec = CountVectorizer(preprocessor=stem5, token_pattern=r"[^ ]+")
X = vec.fit_transform(X_text)
nb = MultinomialNB(alpha=1.0).fit(X, y)
vocab = np.array(vec.get_feature_names_out())
for k, c in enumerate(nb.classes_):
    # soʻz bu sinfda boshqa sinflardagi oʻrtachaga nisbatan qanchalik koʻp uchraydi
    other = np.delete(nb.feature_log_prob_, k, axis=0).mean(axis=0)
    score = nb.feature_log_prob_[k] - other
    top = vocab[np.argsort(-score)[:6]]
    print(f"{c:<7}:", ", ".join(top))

## 3. Morfologik koʻp maʼnolilik: qoidalar asosida tahlil

Oʻzbek tili uchun rus tilidagi `pymorphy3` kabi tayyor tahlilchilar kam. Shuning uchun kichik tahlilchini oʻzimiz yozamiz: oʻzaklar lugʻati (soʻz turkumi bilan) va qoʻshimchalar ketma-ketligi qoidalari. Tahlilchi soʻzning **barcha** mumkin boʻlgan tahlillarini qaytaradi: aynan shu yerda koʻp maʼnolilik koʻrinadi.

In [ ]:
# bu boʻlimda qoʻshimcha kutubxona kerak emas

In [ ]:
# oʻzaklar lugʻati: soʻz -> soʻz turkumlari
LEX = {
    "kitob": ["ot"], "uy": ["ot"], "bola": ["ot"], "maktab": ["ot"], "qishloq": ["ot"],
    "olma": ["ot"], "yoz": ["ot", "feʼl"], "ot": ["ot", "feʼl"], "ol": ["feʼl"], "oʻqi": ["feʼl"],
}
NOUN_SLOTS = [  # tartib: koʻplik -> egalik -> kelishik
    [("lar", "koʻplik")],
    [("imiz", "egalik I sh. koʻp."), ("miz", "egalik I sh. koʻp."), ("ingiz", "egalik II sh. koʻp."), ("ngiz", "egalik II sh. koʻp."),
     ("im", "egalik I sh."), ("m", "egalik I sh."), ("ing", "egalik II sh."), ("ng", "egalik II sh."), ("si", "egalik III sh."), ("i", "egalik III sh.")],
    [("ning", "qaratqich"), ("ni", "tushum"), ("ga", "joʻnalish"), ("ka", "joʻnalish"), ("qa", "joʻnalish"), ("da", "oʻrin-payt"), ("dan", "chiqish")],
]
VERB_SLOTS = [
    [("ma", "inkor")],
    [("di", "oʻtgan zamon"), ("yapti", "hozirgi zamon"), ("gan", "sifatdosh"), ("sh", "harakat nomi"), ("ish", "harakat nomi")],
]
ALTERN = {"g": "k", "gʻ": "q"}   # teragi -> terak, qishlogʻi -> qishloq

def parse_suffixes(rest, slots, i=0):
    # rest qatorini slotlar tartibida qoʻshimchalarga ajratishning barcha usullari
    if rest == "":
        return [[]]
    if i == len(slots):
        return []
    out = parse_suffixes(rest, slots, i + 1)            # bu slot boʻsh
    for suf, gloss in slots[i]:
        if rest.startswith(suf):
            for tail in parse_suffixes(rest[len(suf):], slots, i + 1):
                out.append([(suf, gloss)] + tail)
    return out

def analyses(word):
    word = word.lower()
    res = []
    for cut in range(1, len(word) + 1):
        stem, rest = word[:cut], word[cut:]
        candidates = [stem]
        for a, b in ALTERN.items():
            if stem.endswith(a):
                candidates.append(stem[: -len(a)] + b)
        for root in candidates:
            for pos in LEX.get(root, []):
                slots = NOUN_SLOTS if pos == "ot" else VERB_SLOTS
                for sufs in parse_suffixes(rest, slots):
                    res.append((root, pos, sufs))
    return res

for w in ["olma", "yozda", "yozdi", "otdan", "otdi", "kitoblarimizda", "qishlogʻimizga", "oʻqiyapti"]:
    print(w)
    for root, pos, sufs in analyses(w):
        parts = " + ".join([root] + [s for s, _ in sufs])
        gl = ", ".join(g for _, g in sufs) or "—"
        print(f"   {parts:<24} [{pos}] {gl}")

Bitta otdan nechta shakl hosil boʻladi? Laboratoriyadagi konstruktorni takrorlaymiz: son × egalik × kelishik.

In [ ]:
def is_vowel_end(s):
    return s.endswith(("a", "e", "i", "o", "u", "oʻ"))

POSS = [("", ""), ("im", "m"), ("ing", "ng"), ("i", "si"), ("imiz", "miz"), ("ingiz", "ngiz")]
CASES = ["", "ning", "ni", "ga", "da", "dan"]

def make(root, pl, poss, case):
    w = root + ("lar" if pl else "")
    c_suf, v_suf = poss
    if c_suf:
        if is_vowel_end(w):
            w += v_suf
        else:
            if not pl and len(root) > 3 and w[-1] in "kq":
                w = w[:-1] + ("g" if w[-1] == "k" else "gʻ")
            w += c_suf
    if case == "ga" and w.endswith("k"):
        case = "ka"
    elif case == "ga" and w.endswith("q"):
        case = "qa"
    return w + case

forms = sorted({make("qishloq", pl, ps, cs) for pl in (0, 1) for ps in POSS for cs in CASES})
print(len(forms), "ta shakl, masalan:", ", ".join(forms[:12]), "…")

## 4. Tipologiya raqamlarda

Bir xil qisqa matn uch tilda. Oʻzbek tili (agglyutinativ) grammatikani qoʻshimchalar bilan ifodalaydi, shuning uchun unda soʻzlar kamroq, soʻzlarning oʻzi esa uzunroq. Ingliz tili (analitik) yordamchi soʻzlardan foydalanadi, shuning uchun soʻzlar koʻproq va qisqaroq.

In [ ]:
texts = {
    "ru": [
        "Мы живём в большом доме.",
        "В наших домах тепло и светло.",
        "Дети читают книги в школе.",
        "Учитель дал ученикам новые книги.",
        "Из окна нашего дома видны горы.",
        "Летом мы поедем в деревню к бабушке.",
    ],
    "uz": [
        "Biz katta uyda yashaymiz.",
        "Uylarimizda issiq va yorugʻ.",
        "Bolalar maktabda kitob oʻqiyapti.",
        "Oʻqituvchi oʻquvchilarga yangi kitoblar berdi.",
        "Uyimizning derazasidan togʻlar koʻrinadi.",
        "Yozda biz qishloqqa buvimiznikiga boramiz.",
    ],
    "en": [
        "We live in a big house.",
        "It is warm and bright in our houses.",
        "The children are reading books at school.",
        "The teacher gave the students new books.",
        "The mountains can be seen from the window of our house.",
        "In summer we will go to the village to see our grandmother.",
    ],
}
labels = {"ru": "rus", "uz": "oʻzbek", "en": "ingliz"}
import matplotlib.pyplot as plt

stats = {}
for lang, sents in texts.items():
    words = [w for s in sents for w in tokens(s)]
    stats[lang] = (len(words), np.mean([len(w) for w in words]))
    print(f"{labels[lang]:<11} soʻz: {len(words):3d}   soʻzning oʻrtacha uzunligi: {np.mean([len(w) for w in words]):.1f}")

fig, ax = plt.subplots(1, 2, figsize=(9, 3))
names = [labels[k] for k in stats]
ax[0].bar(names, [v[0] for v in stats.values()], color="#2B55C9")
ax[0].set_title("Matndagi soʻzlar soni")
ax[1].bar(names, [v[1] for v in stats.values()], color="#17907E")
ax[1].set_title("Soʻzning oʻrtacha uzunligi, harf")
plt.tight_layout()
plt.show()

## 5. Topshiriqlar

1. `train` roʻyxatiga yangi maʼno uchun 3–4 ta gap qoʻshing (masalan, «koʻk» = yosh, xom: «koʻk qovun») va 2-boʻlimni qayta ishga tushiring. «Koʻk juda chiroyli edi» gapi uchun entropiya qanday oʻzgardi?
2. «Soʻzlarning oʻzi» va «dastlabki 5 harf» modellarining test gaplaridagi bashoratlarini solishtiring. Nima uchun oʻzbek tilida 5 harfga qisqartirish deyarli yordam bermadi («koʻkda» → «koʻkd»)? Qoʻshimcha topshiriq: 3-boʻlimdagi `analyses()` tahlilchisi yordamida har bir soʻzni oʻzagiga keltiradigan `preprocessor` yozing va natijani solishtiring.
3. 3-boʻlimdagi `LEX` lugʻatiga «yur» (feʼl) va «kun» (ot) oʻzaklarini qoʻshing va kamida ikkita tahlilga ega boʻlgan uchta soʻz toping.
4. 4-boʻlimdagi matnlarda 8 harfdan uzun soʻzlar ulushini rus, oʻzbek va ingliz tillari uchun hisoblang. Natijani tipologiya bilan bogʻlang.

In [ ]:
# Kodingizni shu yerga yozing

In [ ]:
# Kodingizni shu yerga yozing

In [ ]:
# Kodingizni shu yerga yozing

In [ ]:
# Kodingizni shu yerga yozing